# TNBC miRNA–mRNA GAT study, phase 3 (notebook 1 of 2): planted control, variant selection, learning-curve repeats

**Post-hoc, added after the frozen study.** None of this was in the plan frozen after exploration (tag `study-frozen-v1`).
The settings and decision rules are in `kaggle/phase3_config.py`, committed before any phase-3 study fit ran. Nothing in
`study/` is changed: the frozen code is imported, and `pytest study/tests` checks the frozen-file hashes.

| Part | What |
|---|---|
| A | Planted-signal positive control: frozen joint_mirdb / joint_nocross / joint_rewired (paper config) + routed XGBoost and elastic net on a semi-synthetic task readable only along miRDB edges, δ ∈ {0, 0.5, 1, 2}; repeat 0, 5 outer folds × 3 seeds |
| B | Edge-forcing variant selection (xfirst, gated, selfdrop q=0.25/0.5): planted task scored on the **inner-validation split** + real-data inner-validation loss. Writes `selected_variant.json`. No outer test fold is used. |
| C | Learning curve 25% / 50% for joint_mirdb, joint_nocross (+ XGBoost / elastic net) on repeats 1–4 |

The `plan` cell prints the exact fit counts.

**Settings:** Internet **on**, accelerator **None (CPU)**.

**Smoke test first** (`SMOKE_TEST = True`), then `SMOKE_TEST = False` and *Save Version → Save & Run All*.

**Resume:** every fit is one CSV, written atomically, never refit. If the session ends early, attach this notebook's
previous output (*Add Input → Your Work*) and run again; `restore` copies finished fits back.

**After it finishes:** download `phase3_output/selected_variant.json` and give it to Capy (or commit it yourself as
`kaggle/phase3_frozen/selected_variant.json` on `kaggle-phase3`). Notebook 2 refuses to run until that file is committed.

In [ ]:
import os, subprocess, time

SMOKE_TEST = True          # True: one fold, one seed per arm type and a runtime estimate. False: everything.
WORKERS = os.cpu_count()   # parallel fits; each fit is single-threaded (frozen setting)
TIME_BUDGET_H = 11.0       # Kaggle sessions stop at 12 h
1.0_H = 1.0        # kept for baselines, analysis and the zip; no new GAT fit starts after budget - reserve

BRANCH = 'kaggle-phase3'
REPO_URL = 'https://github.com/heramn/TNBC-miRNA-mRNA-GNN.git'
SRC = '/tmp/TNBC-miRNA-mRNA-GNN'
PY = '/tmp/p3env/bin/python'
WORK = '/kaggle/working'
T0 = time.time()
DEADLINE = T0 + (TIME_BUDGET_H - 1.0_H) * 3600
os.environ.update(STUDY_RUNS=f'{WORK}/phase3_runs', PHASE3_OUT=f'{WORK}/phase3_output', STUDY_CACHE='/tmp/study_cache',
                  OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')


def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() and check:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

SM = '--smoke' if SMOKE_TEST else ''
print('workers:', WORKERS, '| smoke test:', SMOKE_TEST)

## 1. Clone the repository at `kaggle-phase3`

In [ ]:
sh(f'rm -rf {SRC} && git clone -q -b {BRANCH} {REPO_URL} {SRC} && git -C {SRC} log -1 --format="%h %s"')

## 2. Python environment

The same isolated Python 3.12 venv as phase 2, with phase 1's pins: torch **2.15.0+cpu**, torch_geometric **2.8.1**,
scikit-learn **1.7.2** (1.8 changes the grouped splits). Kaggle's preinstalled CUDA torch is left alone.

In [ ]:
sh('pip install -q uv')
sh('uv venv -q --python 3.12 /tmp/p3env')
sh(f'uv pip install -q --python {PY} numpy==2.5.5 pandas==3.1.1 scipy==1.18.1 scikit-learn==1.7.2 statsmodels==0.15.0 '
   'matplotlib==3.11.2 xgboost==3.4.1 pydeseq2==0.6.0 anndata==0.13.4 torch_geometric==2.8.1 pytest')
sh(f'uv pip install -q --python {PY} torch==2.15.0 --index-url https://download.pytorch.org/whl/cpu')
sh(f'{PY} -c "import sys, torch, torch_geometric, sklearn, numpy; print(sys.version.split()[0], torch.__version__, '
   'torch_geometric.__version__, sklearn.__version__, numpy.__version__)"')

## 3. Data, checkpoint restore, frozen-file tests, plan

`data` checks the GEO inputs and miRDB by sha256 (as phase 2) and copies the phase-1 configuration selections.
`check-plant` prints, per fold and strength, the planted hubs, the largest single-feature class signal (η², should stay
near 0 at every δ) and the balanced accuracy of a logistic model on the two true hub set means (chance 0.33): the
ceiling an edge-using model could reach.

In [ ]:
os.chdir(SRC)
sh(f'{PY} kaggle/phase3.py data')
sh(f'{PY} kaggle/phase3.py restore')
sh(f'{PY} -m pytest -q -p no:cacheprovider study/tests')
sh(f'{PY} kaggle/phase3.py plan')
sh(f'{PY} kaggle/phase3.py check-plant')

## 4. Part A: planted-signal positive control

In [ ]:
sh(f'{PY} kaggle/phase3.py run --part A {SM} --workers {WORKERS} --deadline {DEADLINE:.0f}')
sh(f'{PY} kaggle/phase3.py baselines --part A {SM} --workers {WORKERS}')

## 5. Part B: edge-forcing variant selection (inner validation only)

Four candidates × {miRDB, no-cross, rewired} edges × δ ∈ {0, 1} on the planted task, scored on the inner-validation
split, plus each candidate with miRDB edges on the real labels (inner-validation loss). `select` applies the rule in
`phase3_config.py` and writes `phase3_output/selected_variant.json`.

In [ ]:
sh(f'{PY} kaggle/phase3.py run --part B {SM} --workers {WORKERS} --deadline {DEADLINE:.0f}')
if not SMOKE_TEST:
    rc = sh(f'{PY} kaggle/phase3.py select', check=False)
    print('selection', 'written' if rc == 0 else f'not complete (exit {rc}); attach this output and rerun')

## 6. Part C: learning-curve repeats

In [ ]:
sh(f'{PY} kaggle/phase3.py run --part C {SM} --workers {WORKERS} --deadline {DEADLINE:.0f}')
sh(f'{PY} kaggle/phase3.py baselines --part C {SM} --workers {WORKERS}')
if SMOKE_TEST:
    sh(f'{PY} kaggle/phase3.py estimate --workers {WORKERS}')

## 7. Analysis and RESULTS rows (full mode)

In [ ]:
if SMOKE_TEST:
    print('smoke test: analysis skipped')
else:
    for part in ('A', 'C'):
        rc = sh(f'{PY} kaggle/phase3.py analyze --part {part}', check=False)
        print('part', part, 'analysis', 'done' if rc == 0 else f'failed (exit {rc})')
    sh(f'{PY} kaggle/phase3.py report', check=False)
    sh(f'{PY} kaggle/phase3.py plan')
    sel = f'{WORK}/phase3_output/selected_variant.json'
    if os.path.exists(sel):
        print(open(sel).read())

## 8. Zip the outputs

`phase3_runs/` is the checkpoint (every per-run CSV); `phase3_output/` has the tables, `selected_variant.json`, the fit log and `RESULTS_phase3_rows.md`.

In [ ]:
import shutil, zipfile
os.chdir(WORK)
z = shutil.make_archive(f'{WORK}/tnbc_phase3a', 'zip', WORK, '.')
with zipfile.ZipFile(z) as zf:
    names = [n for n in zf.namelist() if n.startswith('phase3_runs/') and n.endswith('.csv')]
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB,', len(names), 'per-run CSVs')
rows = f'{WORK}/phase3_output/RESULTS_phase3_rows.md'
if os.path.exists(rows):
    print(open(rows).read())
print(f'total time {(time.time() - T0) / 3600:.2f} h')